# Biocomputing with Brainoware
### Procedure 1 - Human cortical organoid generation and Brainoware hardware integration
#### 1. Spike sorting with Kilosort2 and functional connectivity calculation (local version)
**Author: Hongwei Cai, Huiyu Chu**  
**Date: June 6, 2026**  
**Description**: This notebook provides a step-by-step guide for spike sorting using Kilosort2 in local environment, specifically based on "Network Scan" result of cortical organoid attached on the MaxOne MEA. It covers dependency installation, raw data processing, sorting, quality metric computation, curation, and functional connectivity analysis.


In [1]:
import glob
import re
import os
import matplotlib.pyplot as plt
import numpy as np  
import pandas as pd

import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.comparison as sc
import spikeinterface.widgets as sw
from spikeinterface.postprocessing import compute_unit_locations
from spikeinterface import extract_waveforms
from matplotlib.pyplot import figure
import networkx as nx
ss.Kilosort2Sorter.check_compiled = classmethod(lambda cls: False)
%matplotlib inline
# encoding: utf-8

d:\miniconda\envs\general\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set Kilosort2 path
ss.Kilosort2Sorter.set_kilosort2_path('D://Kilosort-2.0')

# Define Kilosort2 parameters
kilo_params = ss.Kilosort2Sorter.default_params()
kilo_params['minFR'] = 0.02
kilo_params['minfr_goodchannels'] = 0.1
kilo_params['detect_threshold'] = 5.5
kilo_params['n_jobs'] = 8
kilo_params

Setting KILOSORT2_PATH environment variable for subprocess calls to: D:\Kilosort-2.0


{'detect_threshold': 5.5,
 'projection_threshold': [10, 4],
 'preclust_threshold': 8,
 'whiteningRange': 32,
 'momentum': [20.0, 400.0],
 'car': True,
 'minFR': 0.02,
 'minfr_goodchannels': 0.1,
 'freq_min': 150,
 'sigmaMask': 30,
 'lam': 10.0,
 'nPCs': 3,
 'ntbuff': 64,
 'nfilt_factor': 4,
 'NT': None,
 'AUCsplit': 0.9,
 'wave_length': 61,
 'keep_good_only': False,
 'skip_kilosort_preprocessing': False,
 'scaleproc': None,
 'save_rez_to_mat': False,
 'delete_tmp_files': ('matlab_files',),
 'delete_recording_dat': False,
 'pool_engine': 'process',
 'n_jobs': 8,
 'chunk_duration': '1s',
 'progress_bar': True,
 'mp_context': None,
 'max_threads_per_worker': 1}

In [3]:


# Base directory
base_path = "./"

# Find all data.raw.h5 files under Network/<number>/
pattern = os.path.join(base_path, 'fc*.raw.h5')
all_files = glob.glob(pattern, recursive=True)

# Loop through files
for file_path in all_files:
    folder = os.path.dirname(file_path)
    sorter_output = os.path.join(folder, 'kilosort2')
    
    if os.path.exists(sorter_output):
        print(f"Skipping {folder} — 'kilosort2' already exists.")
        continue

    try:
        print(f"Processing: {file_path}")
        
        # Load recording
        recording = se.MaxwellRecordingExtractor(file_path)
        recording_signed = spre.unsigned_to_signed(recording)
        # Apply bandpass filter
        recording_filtered = spre.bandpass_filter(recording_signed, freq_min=300, freq_max=6000)
        
        # Run sorter
        sorting = ss.run_sorter(
            sorter_name="kilosort2",
            recording=recording_filtered,
            folder=sorter_output,
            verbose=True,
            **kilo_params
        )
        
        waveform_extractor = extract_waveforms(
            recording=recording_filtered,
            sorting=sorting,
            folder=os.path.join(sorter_output, "waveforms"),
            ms_before=1.0,
            ms_after=2.0,
            max_spikes_per_unit=500,
            overwrite=None,   # not True/False
            n_jobs=8
        )
        # Compute unit locations (default method is 'center_of_mass')
        compute_unit_locations(waveform_extractor)

        # Get the computed locations
        unit_locations = waveform_extractor.load_extension("unit_locations").get_data()

        unit_ids = sorting.unit_ids
        duration_s = recording.get_total_duration()
        
        
        firing_rates = []
        for unit_id in unit_ids:
            spike_train = sorting.get_unit_spike_train(unit_id)
            firing_rate = len(spike_train) / duration_s
            firing_rates.append(firing_rate)

        df_firing = pd.DataFrame({
            "unit_id": unit_ids,
            "firing_rate": firing_rates
        })

        # -----------------------------
        # Spike trains table
        # -----------------------------
        rows = []
        for unit_id in unit_ids:
            spike_times = sorting.get_unit_spike_train(unit_id)
            for t in spike_times:
                rows.append({"unit_id": unit_id, "time": t})

        df_spikes = pd.DataFrame(rows)

        # -----------------------------
        # Unit locations table
        # -----------------------------
        xy_locations = unit_locations[:, :2]
        df_locations = pd.DataFrame(xy_locations, columns=["x", "y"])
        df_locations.insert(0, "unit_id", unit_ids)

        df_merged = pd.merge(df_locations, df_firing, on="unit_id")

        # -----------------------------
        # Plot
        # -----------------------------
        ax1 = df_merged.plot.scatter(
            x="x",
            y="y",
            s=df_merged["firing_rate"] * 70,
            c="unit_id",
            colormap="cividis",
            alpha=0.8
        )
        ax1.set_aspect("equal")
        ax1.figure.set_size_inches(12, 9)
        ax1.set_ylabel("Y (μm)")
        ax1.set_xlabel("X (μm)")

        plt.savefig(os.path.join(sorter_output, "firing_map.svg"), bbox_inches="tight")
        plt.savefig(os.path.join(sorter_output, "firing_map.png"), bbox_inches="tight")
        plt.close()

        # -----------------------------
        # Save outputs
        # -----------------------------
        df_spikes.to_csv(os.path.join(sorter_output, "spike_trains.csv"), index=False)
        df_merged[["unit_id", "x", "y"]].to_csv(
            os.path.join(sorter_output, "neuron_coord.csv"),
            index=False
        )

        print(f"Done: {folder}")

        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")


Processing: .\fc_example_data.raw.h5


d:\miniconda\envs\general\lib\site-packages\spikeinterface\extractors\__init__.py:61: UserWarning: Importing classes at __init__ has been deprecated in favor of only importing function-size wrappers and will be removed in 0.105.0. For developers that prefer working with the class versions of extractors they can be imported from spikeinterface.extractors.extractor_classesFor class MaxwellRecordingExtractor
  warn(dep_msg)


The h5 compression library for Maxwell is already located in D:\IUcodings\MaxOne\compression.dll!


write_binary_recording (workers: 8 processes): 100%|██████████| 301/301 [00:44<00:00,  6.71it/s]


RUNNING SHELL SCRIPT: d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local\kilosort2\sorter_output\run_kilosort2.bat


(general) d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local>d:



(general) d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local>cd d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local\kilosort2\sorter_output 



(general) d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local\kilosort2\sorter_output>matlab -nosplash -wait -r "kilosort2_master('d:\IUcodings\Protocol_speech_recog\github_clean\Stage_1\1_kilosort_functional_connectivity_local\kilosort2\sorter_output', 'D:\Kilosort-2.0')" 

kilosort2 run time 620.91s
The h5 compression library for Maxwell is already located in D:\IUcodings\MaxOne\compression.dll!
####
# extract_waveforms() and WaveformExtr

estimate_sparsity (workers: 8 processes): 100%|██████████| 301/301 [00:40<00:00,  7.37it/s]
d:\miniconda\envs\general\lib\site-packages\spikeinterface\core\basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (no parallelization): 100%|██████████| 20/20 [00:08<00:00,  2.36it/s]


Done: .


# Functional connectivity from STTC

Compute the spike-time tiling coefficient (STTC) directly from the sorting result `df_spikes` (columns `unit_id`, `time` in samples), then draw a functional connectivity map whose node positions come from `df_locations` (columns `unit_id`, `x`, `y`).


In [4]:
# Define STTC matrix computation using Elephant (Cutts & Eglen 2014)
import numpy as np
import neo
import quantities as pq
from elephant.spike_train_correlation import spike_time_tiling_coefficient


def compute_sttc_matrix(spike_trains_s, dt_s, t_start_s, t_stop_s):
    """Symmetric STTC matrix using elephant.spike_time_tiling_coefficient.

    Parameters
    ----------
    spike_trains_s : list of 1D arrays
        Spike times in seconds, one array per unit.
    dt_s : float
        Synchrony window (+/- dt) in seconds.
    t_start_s, t_stop_s : float
        Recording start/stop in seconds (used for the tiling fraction TA/TB).

    Returns
    -------
    np.ndarray (n x n)
        STTC values; diagonal left as NaN (no self-connections), NaN for empty trains.
    """
    n = len(spike_trains_s)
    neo_trains = [
        neo.SpikeTrain(np.sort(np.asarray(st, dtype=float)) * pq.s,
                       t_start=t_start_s * pq.s, t_stop=t_stop_s * pq.s)
        for st in spike_trains_s
    ]
    dt_q = dt_s * pq.s
    sttc_matrix = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(i + 1, n):
            sttc = spike_time_tiling_coefficient(neo_trains[i], neo_trains[j], dt=dt_q)
            sttc_matrix[i, j] = sttc
            sttc_matrix[j, i] = sttc
    return sttc_matrix


In [5]:
# Compute STTC from the in-memory sorting result `df_spikes` (Elephant)
import pandas as pd

# Settings
sampling_rate = 20000          # Hz
dt = 0.030                     # 30 ms coincidence window (seconds)
recording_duration_s = None    # set to the recording length in seconds if known;
                               # otherwise the last spike time is used as t_stop

# Per-unit spike trains (sorted, in seconds), keyed by unit_id
spike_trains = {
    unit: np.sort(group["time"].values.astype(np.float64)) / sampling_rate
    for unit, group in df_spikes.groupby("unit_id")
}

# Fixed unit order used for every matrix / coordinate downstream
selected_unit_ids = sorted(spike_trains.keys())
n_units = len(selected_unit_ids)

# List of spike-train arrays (same order as selected_unit_ids)
spike_trains_list = [spike_trains[uid] for uid in selected_unit_ids]

# Recording span in seconds (t_stop must be >= the last spike)
last_spike_s = float(df_spikes["time"].max()) / sampling_rate
duration = max(recording_duration_s or 0.0, last_spike_s)

# Run STTC (elephant.spike_time_tiling_coefficient for every unit pair)
sttc_fast = compute_sttc_matrix(spike_trains_list, dt, 0.0, duration)

# Labelled DataFrame (rows/cols are unit_id)
sttc_df = pd.DataFrame(sttc_fast, index=selected_unit_ids, columns=selected_unit_ids)
sttc_df


,0,1,2,3,4,5,6,7,8,9,...,1367,1368,1369,1370,1371,1372,1373,1374,1375,1376
0,NaN,-0.014619,-0.037918,-0.002005,-0.002002,-0.018356,-0.061058,-0.006699,-0.000700,-0.000500,...,0.109708,-0.061822,0.137094,0.060803,-0.032053,-0.040911,0.276053,0.068527,0.147746,-0.040320
1,-0.014619,NaN,0.453944,0.341400,0.044111,-0.023927,0.198777,-0.020718,0.113302,0.241127,...,0.061657,0.168057,-0.037231,-0.090920,0.050069,-0.051785,0.072729,-0.029855,0.047578,-0.010350
2,-0.037918,0.453944,NaN,0.140098,0.107627,0.026933,0.178491,-0.017092,-0.038018,0.221663,...,0.069425,0.255409,0.060431,-0.023149,0.022108,-0.015567,0.085802,0.021478,0.252373,0.082845
3,-0.002005,0.341400,0.140098,NaN,-0.003407,-0.019762,0.083227,-0.008104,-0.002105,-0.001905,...,-0.006692,0.024438,-0.035424,-0.112053,0.173940,-0.042316,0.051841,0.039223,-0.025734,-0.041726
4,-0.002002,0.044111,0.107627,-0.003407,NaN,-0.019758,0.112750,-0.008101,-0.002102,-0.001902,...,0.051062,0.023667,0.055422,-0.028813,-0.033455,-0.042313,0.051092,0.008701,0.098667,0.077670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1372,-0.040911,-0.051785,-0.015567,-0.042316,-0.042313,0.083076,0.009892,0.052680,-0.041011,-0.040811,...,0.022973,0.030458,0.056067,0.047692,0.032966,NaN,0.045552,0.073994,-0.012621,0.107633
1373,0.276053,0.072729,0.085802,0.051841,0.051092,0.121035,0.127674,-0.006264,0.503362,0.176727,...,0.098683,0.060204,0.037412,0.002863,0.082410,0.045552,NaN,0.010492,0.050396,0.063961
1374,0.068527,-0.029855,0.021478,0.039223,0.008701,0.041721,0.017932,0.105954,-0.103412,-0.103212,...,0.013101,0.018723,0.015068,0.185440,0.029741,0.073994,0.010492,NaN,0.012135,0.045119
1375,0.147746,0.047578,0.252373,-0.025734,0.098667,0.014524,0.089587,-0.019541,-0.024429,-0.024229,...,0.014400,0.110565,0.043903,0.011423,-0.007486,-0.012621,0.050396,0.012135,NaN,0.081089


## Build the thresholded adjacency matrix


In [6]:
# Plot / threshold parameters
color_plot = 5
color_shift = 3
color_total = color_plot + color_shift
width_total = 5
dw = 1
thred_c = 0.3        # adjacency cutoff: STTC below this -> no edge
thred_delta = 0.25
thred_plot = 0.35    # only edges above this are drawn
color_map = 'RdYlGn_r'
node_color = 'mediumseagreen'


In [7]:
# Adjacency matrix: replace NaN with 0 and drop weak / negative couplings
adj_matrix = np.nan_to_num(sttc_fast, nan=0.0)
adj_matrix[adj_matrix < thred_c] = 0


## Align unit locations to the STTC unit order


In [8]:
# Map each STTC unit to its (x, y) from `df_locations`, in selected_unit_ids order.
# This guarantees node i of the graph sits at the location of selected_unit_ids[i].
coord_lookup = df_locations.set_index("unit_id")[["x", "y"]]
np_coord = coord_lookup.loc[selected_unit_ids].to_numpy()
np_coord


array([[2932.35535646, 1444.96676883],
       [2999.66227228, 1131.97990507],
       [2992.11657557, 1106.37572514],
       ...,
       [1208.72928066, 1053.81919834],
       [2810.67085106,  818.23764542],
       [2386.38138248, 1699.75539313]])

In [9]:
# Build the graph; node index i corresponds to selected_unit_ids[i]
nx_adj = nx.from_numpy_array(adj_matrix)
pos_ = np_coord                       # use real electrode coordinates as layout
sorted_edges = sorted(nx_adj.edges(data=True), key=lambda x: x[2]['weight'])


## Draw the functional connectivity map


In [15]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mc
import plotly.graph_objects as go


In [16]:
cmap = plt.get_cmap(color_map)


In [17]:
widths = np.arange(0, dw * width_total, dw) + 1
for i in range(0, width_total):
    widths[i] = (widths[i] / 2) ** 2 + 1

colors_with_alpha = []
for i in range(color_total):
    rgba = list(cmap(i))
    alpha = (i + 1) / color_total      # ramp transparency with strength
    rgba[3] = alpha
    colors_with_alpha.append(tuple(rgba))


def rgba_to_plotly_string(rgba_tuple):
    r, g, b, a = rgba_tuple
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a:.3f})"


color_scale = [rgba_to_plotly_string(rgba) for rgba in colors_with_alpha]


In [18]:
def make_edge(x, y, text, width, colors):
    """Scatter trace for a single edge between two nodes."""
    return go.Scatter(x=x,
                      y=y,
                      line=dict(width=width, color=colors),
                      hoverinfo='text',
                      text=([text]),
                      mode='lines')


In [20]:
# Build edge traces (strongest-on-top via sorted_edges)
edge_trace = []
for edge in sorted_edges:
    if edge[2]['weight'] > thred_plot:
        char_1 = edge[0]
        char_2 = edge[1]

        x0, y0 = pos_[char_1]
        x1, y1 = pos_[char_2]

        text = None

        # Map STTC in (thred_plot, 1] linearly onto the full color scale and width range
        frac = min(max((edge[2]['weight'] - thred_plot) / (1 - thred_plot), 0.0), 1.0)
        color_num = min(int(frac * color_total), color_total - 1)
        width_num = min(int(frac * width_total), width_total - 1)

        trace = make_edge([x0, x1, None], [y0, y1, None], text,
                          widths[width_num], color_scale[color_num])
        edge_trace.append(trace)

# Node trace: position from df_locations, size from total connectivity, label = unit_id
node_trace = go.Scatter(x=[], y=[], text=[],
                        textposition="top center",
                        textfont_size=10,
                        mode='markers+text',
                        hoverinfo='text',
                        marker=dict(color=[], size=[], line=None))

for node in nx_adj.nodes():
    x, y = pos_[node]
    node_trace['x'] += tuple([x])
    node_trace['y'] += tuple([y])
    node_trace['marker']['color'] += tuple([node_color])
    node_trace['marker']['size'] += tuple([2 * (sum(adj_matrix[:, node])) ** (1 / 3)])
    #node_trace['text'] += tuple(['<b>' + str(selected_unit_ids[node]) + '</b>'])

layout = go.Layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    width=1000,
    height=800
)

fig = go.Figure(layout=layout)
for trace in edge_trace:
    fig.add_trace(trace)
fig.add_trace(node_trace)

fig.update_layout(showlegend=False)
fig.update_xaxes(showticklabels=True)
fig.update_yaxes(showticklabels=True, scaleanchor="x", scaleratio=1,
                 autorange="reversed")   # y = 0 at the top, as on the MEA

fig.show()
